In [347]:
import pandas as pd
import numpy as np
import random
import math

# Recipe class


In [348]:
import pandas as pd

recipes_df = pd.read_csv("./data/recipes.csv")

def gettransitionModel(recipes_df):
    result = {}

    for _, row in recipes_df.iterrows():
        meal_type = row["type"]
        meal_name = row["Name"]

        meal_info = (
            row["Category"],
            row["total_price"],
            row["provided_calories"],
            row["provided_protein"],
            row["provided_carbs"],
            row["provided_fat"]
        )

        if meal_type not in result:
            result[meal_type] = {}

        result[meal_type][meal_name] = meal_info

    return result

transition_model = gettransitionModel(recipes_df)

In [349]:
NUTRI_PREFERENCE_MAP = {
    "maintain":       {"protein": 0.30, "fat": 0.25, "carbs": 0.45},
    "gain":   {"protein": 0.40, "fat": 0.25, "carbs": 0.35},
    "loss":       {"protein": 0.25, "fat": 0.35, "carbs": 0.40},
}

# Node class

In [350]:
class MealPlannerState:
    def __init__(self, day_number, meal_type, meal, remaining_budget, today_calorie_use, today_protein_use, today_carbs_use, today_fat_use, used_meals = None):
        self.day = day_number
        self.meal_type = meal_type
        self.meal = meal
        self.remaining_budget = remaining_budget
        self.today_calorie_use = today_calorie_use
        self.today_protein_use = today_protein_use
        self.today_carbs_use = today_carbs_use
        self.today_fat_use = today_fat_use
        self.used_meals = used_meals
        
    def __eq__(self, other):
        return isinstance(other, MealPlannerState) and \
            self.day == other.day and \
            self.meal_type == other.meal_type and \
            self.meal == other.meal

    def __hash__(self):
        return hash((self.day, self.meal_type, self.meal))
        

In [351]:
class Node:    
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = (parent.g + cost) if parent else 0
        self.f = self.g + heuristic
        self.depth = 0 if parent is None else parent.depth + 1

    def path(self):
        node = self
        actions = []
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        actions.reverse()
        return actions

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)


# Problem class

In [352]:
class MealPlanningProblem:

    def __init__(self, transition_model, TDEE, total_budget, num_days=7, macroNutri_preference="maintain"):
        self.transition_model = transition_model
        self.total_budget = total_budget
        self.TDEE = TDEE
        self.num_days = num_days
        self.meals_per_day = 3  # Breakfast, Lunch, Dinner
        self.total_slots = num_days * self.meals_per_day
        self.meal_types = ['Breakfast', 'Lunch', 'Dinner']

        self.meal_type_weights = {
            'Breakfast': 0.25,
            'Lunch': 0.40,
            'Dinner': 0.35
        }

        self.macro_nutrients_ratios = NUTRI_PREFERENCE_MAP[macroNutri_preference]

        self.initial_state = MealPlannerState(0, 0, None, total_budget, 0, 0, 0, 0, set())
        
        self._avg_price = recipes_df['total_price'].mean()

    def _get_remaining_days(self, state):
        current_slot = state.day * self.meals_per_day + state.meal_type + 1
        remaining_slots = self.total_slots - current_slot
        return math.ceil(remaining_slots / self.meals_per_day)

    def _calculate_totals(self, recipes_list):
        total_cost = 0
        total_calories = 0
        
        i = 0
        for meal_name in recipes_list:
            recipe = self.transition_model[self.meal_types[i]][meal_name]
            total_cost += recipe[1]
            total_calories += recipe[2]
            
            i = (i+1)%3

        return total_cost, total_calories

    def is_goal(self, node):
        state = node.state
        
        if state.day < self.num_days:
            return False
        
        recipes_chosen = node.path()
        
        if len(recipes_chosen) != self.total_slots:
            return False
        
        total_cost, total_calories = self._calculate_totals(recipes_chosen)
        
        goal_calories = self.TDEE * self.num_days
        calorie_tolerance = goal_calories * 0.1
        calories_ok = abs(total_calories - goal_calories) <= calorie_tolerance or True
        
        budget_ok = total_cost <= self.total_budget
        
        return calories_ok and budget_ok

    def expand_node(self, node, use_cost=True, use_heuristic=False):
        state = node.state
        
        if state.day >= self.num_days:
            return []
        
        children = []
        valid_actions = self.transition_model[self.meal_types[node.state.meal_type]]
        
        for meal_name in valid_actions:

            if meal_name in state.used_meals:
                continue

            recipe = self.transition_model[self.meal_types[node.depth%3]][meal_name]
            
            action_cost = self.calculate_cost(node.state, meal_name) if use_cost else 0
            
            new_meal_idx = (state.meal_type + 1) % self.meals_per_day
            new_day = state.day if new_meal_idx > 0 else state.day + 1
            new_remaining_budget = state.remaining_budget - recipe[1]
            new_calorie_use = 0 if new_meal_idx == 0 else state.today_calorie_use + recipe[2]
            new_protein_use = 0 if new_meal_idx == 0 else state.today_protein_use + recipe[3]
            new_carbs_use = 0 if new_meal_idx == 0 else state.today_carbs_use + recipe[4]
            new_fat_use = 0 if new_meal_idx == 0 else state.today_fat_use + recipe[5]
            
            if new_day % 15 == 0 and new_day != 0:
                new_used_meals = set({})
            else:
                new_used_meals = set(state.used_meals)

            new_used_meals.add(meal_name)
            
            new_state = MealPlannerState(new_day, new_meal_idx, meal_name, new_remaining_budget,
            new_calorie_use, new_protein_use, new_carbs_use, new_fat_use, new_used_meals)
            
            heuristic = self.calculate_heuristic(new_state) if use_heuristic else 0
            
            child = Node(new_state, parent=node, action=meal_name, cost=action_cost, heuristic=heuristic)
            children.append(child)
        
        return children

    def calculate_cost(self, state, meal_name=None):
        if meal_name is None:
            meal_name = state.meal
        
        if meal_name is None:
            return 0
        
        recipe = self.transition_model[self.meal_types[state.meal_type]][meal_name]
        meal_type = self.meal_types[state.meal_type]
        
        remaining_days = self._get_remaining_days(state)
        
        allocated_price = self.meal_type_weights[meal_type] * (state.remaining_budget / max(1, remaining_days))
        allocated_calories = self.meal_type_weights[meal_type] * self.TDEE 

        allocated_protein = self.meal_type_weights[meal_type] * self.TDEE * self.macro_nutrients_ratios["protein"]
        allocated_carbs = self.meal_type_weights[meal_type] * self.TDEE * self.macro_nutrients_ratios["carbs"]
        allocated_fat = self.meal_type_weights[meal_type] * self.TDEE * self.macro_nutrients_ratios["fat"]

        price_deviation = abs(recipe[1] - allocated_price)
        calorie_deviation = abs(recipe[2] - allocated_calories)
        protein_deviation = abs(recipe[3] - allocated_protein)
        carbs_deviation = abs(recipe[4] - allocated_carbs)
        fat_deviation = abs(recipe[5] - allocated_fat)

        normalized_price_dev = price_deviation / allocated_price if allocated_price > 0 else 0
        normalized_cal_dev = calorie_deviation / allocated_calories if allocated_calories > 0 else 0
        normalized_protein_dev = protein_deviation / allocated_protein if allocated_protein > 0 else 0
        normalized_carbs_dev = carbs_deviation / allocated_carbs if allocated_carbs > 0 else 0
        normalized_fat_dev = fat_deviation / allocated_fat if allocated_fat > 0 else 0

        total_cost = (normalized_price_dev + normalized_cal_dev + normalized_protein_dev + normalized_carbs_dev + normalized_fat_dev) / 5.0
        
        return total_cost


    def calculate_heuristic(self, state):
        remaining_slots = self.total_slots - (state.day * self.meals_per_day + state.meal_type)
        if remaining_slots <= 0:
            return 0

        ideal_spend = state.remaining_budget / remaining_slots

        # price_gap = abs(self._avg_price - ideal_spend) / self._avg_price
        calorie_gap = abs(state.today_calorie_use - self.TDEE) / self.TDEE
        protein_gap = abs(state.today_protein_use - (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["protein"])) / (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["protein"]) if (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["protein"]) > 0 else 0
        carbs_gap = abs(state.today_carbs_use - (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["carbs"])) / (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["carbs"]) if (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["carbs"]) > 0 else 0
        fat_gap = abs(state.today_fat_use - (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["fat"])) / (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["fat"]) if (self.meal_type_weights[self.meal_types[state.meal_type]] * self.TDEE * self.macro_nutrients_ratios["fat"]) > 0 else 0

        return ( calorie_gap + protein_gap + carbs_gap + fat_gap) / 4.0

        # return (price_gap + calorie_gap) / 2.0


# Search class

In [353]:
import queue

class AstarSearch:
    def __init__(self,problem):
        self.problem = problem
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self._get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, True, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)

    def _get_solution_path(self, solution_node):
        path = solution_node.path()
        path = [tuple([
                path[i],
                path[i+1],
                path[i+2]
            ]) for i in range(0, len(path), 3)]

        return path



In [354]:
class GreedySearch:
    def __init__(self,problem):
        self.problem = problem
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self._get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, False, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)

    def _get_solution_path(self, solution_node):
        path = solution_node.path()
        path = [tuple([
                path[i],
                path[i+1],
                path[i+2]
            ]) for i in range(0, len(path), 3)]

        return path



In [ ]:
def test_search(TDEE, budget, days, strat = "A*"):
    problem = MealPlanningProblem(TransitionModel, TDEE, budget, days)

    S = AstarSearch(problem) if strat == "A*" else GreedySearch(problem)

    solution = S.search()

    if solution is None:
        print("couldn't find a suitable plan")
        return

    # Helper function to find meal info by name
    def find_meal(meal_name):
        for meal_type in TransitionModel:
            if meal_name in TransitionModel[meal_type]:
                return TransitionModel[meal_type][meal_name]
        return None
    
    # Extract recipes from solution and calculate statistics
    recipe_data = []
    for day_meals in solution:
        for meal_name in day_meals:
            meal_info = find_meal(meal_name)
            if meal_info:
                recipe_data.append(meal_info)
    
    # Calculate totals (note: protein, carbs, fat are stored as calorie contributions)
    total_cost = sum(r[1] for r in recipe_data)  # total_price at index 1
    total_calories = sum(r[2] for r in recipe_data)  # provided_calories at index 2
    total_protein_cal = sum(r[3] for r in recipe_data)  # provided_protein (in calories) at index 3
    total_carbs_cal = sum(r[4] for r in recipe_data)  # provided_carbs (in calories) at index 4
    total_fat_cal = sum(r[5] for r in recipe_data)  # provided_fat (in calories) at index 5
    
    # Convert calorie contributions to grams
    total_protein_g = total_protein_cal / 4
    total_carbs_g = total_carbs_cal / 4
    total_fat_g = total_fat_cal / 9
    
    # Calculate macronutrient ratios as percentages of total calories
    actual_protein_ratio = (total_protein_cal / total_calories * 100) if total_calories > 0 else 0
    actual_carbs_ratio = (total_carbs_cal / total_calories * 100) if total_calories > 0 else 0
    actual_fat_ratio = (total_fat_cal / total_calories * 100) if total_calories > 0 else 0
    
    # Get target macronutrient ratios
    target_ratios = NUTRI_PREFERENCE_MAP.get("maintain", {"protein": 0.30, "fat": 0.25, "carbs": 0.45})
    target_protein_pct = target_ratios["protein"] * 100
    target_carbs_pct = target_ratios["carbs"] * 100
    target_fat_pct = target_ratios["fat"] * 100
    
    # Calculate deviations
    expected_calories = TDEE * days
    calorie_deviation = total_calories - expected_calories
    calorie_deviation_pct = (calorie_deviation / expected_calories) * 100 if expected_calories > 0 else 0
    
    budget_remaining = budget - total_cost
    budget_deviation_pct = (budget_remaining / budget) * 100 if budget > 0 else 0
    
    # Macronutrient deviations
    protein_dev = actual_protein_ratio - target_protein_pct
    carbs_dev = actual_carbs_ratio - target_carbs_pct
    fat_dev = actual_fat_ratio - target_fat_pct
    
    # Daily breakdown
    daily_costs = []
    daily_calories = []
    daily_protein = []
    daily_carbs = []
    daily_fat = []
    for i, day_meals in enumerate(solution):
        day_cost = 0
        day_calories = 0
        day_protein_cal = 0
        day_carbs_cal = 0
        day_fat_cal = 0
        for meal_name in day_meals:
            meal_info = find_meal(meal_name)
            if meal_info:
                day_cost += meal_info[1]  # total_price
                day_calories += meal_info[2]  # provided_calories
                day_protein_cal += meal_info[3]  # provided_protein (calories)
                day_carbs_cal += meal_info[4]  # provided_carbs (calories)
                day_fat_cal += meal_info[5]  # provided_fat (calories)
        daily_costs.append(day_cost)
        daily_calories.append(day_calories)
        daily_protein.append(day_protein_cal) 
        daily_carbs.append(day_carbs_cal) 
        daily_fat.append(day_fat_cal)  
    
    # Unique meals
    unique_meals = set(meal_name for day_meals in solution for meal_name in day_meals)
    
    # Print results
    print("="*60)
    print("A* MEAL PLAN RESULTS")
    print("="*60)
    print(f"\n PARAMETERS:")
    print(f"  Daily TDEE target: {TDEE} cal")
    print(f"  Total budget: {budget} DZD")
    print(f"  Duration: {days} days")
    
    print(f"\n COST ANALYSIS:")
    print(f"  Total cost: {total_cost:.2f} DZD")
    print(f"  Budget remaining: {budget_remaining:.2f} DZD ({budget_deviation_pct:.1f}%)")
    print(f"  Average daily cost: {total_cost/days:.2f} DZD")
    
    print(f"\n CALORIE ANALYSIS:")
    print(f"  Total calories: {total_calories:.0f} cal")
    print(f"  Expected calories: {expected_calories} cal")
    print(f"  Deviation: {calorie_deviation:+.0f} cal ({calorie_deviation_pct:+.1f}%)")
    print(f"  Average daily calories: {total_calories/days:.0f} cal")
    
    print(f"\n MACRONUTRIENT DISTRIBUTION:")
    print(f"  Protein: {total_protein_g:.0f} ({actual_protein_ratio:.1f}%) [Target: {target_protein_pct:.0f}%] {protein_dev:+.1f}%")
    print(f"  Carbs:   {total_carbs_g:.0f} ({actual_carbs_ratio:.1f}%) [Target: {target_carbs_pct:.0f}%] {carbs_dev:+.1f}%")
    print(f"  Fat:     {total_fat_g:.0f} ({actual_fat_ratio:.1f}%) [Target: {target_fat_pct:.0f}%] {fat_dev:+.1f}%")
    print(f"\n  Daily Avg Macros:")
    print(f"    Protein: {total_protein_g/days:.0f}g | Carbs: {total_carbs_g/days:.0f}g | Fat: {total_fat_g/days:.0f}g")
    
    print(f"\n DAILY BREAKDOWN:")
    for day in range(days):
        status = "✓" if abs(daily_calories[day] - TDEE) / TDEE < 0.1 else "⚠"
        day_protein_ratio = ((daily_protein[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        day_carbs_ratio = ((daily_carbs[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        day_fat_ratio = ((daily_fat[day] / daily_calories[day]) * 100) if daily_calories[day] > 0 else 0
        print(f"  Day {day+1}: {daily_costs[day]:.2f} DZD, {daily_calories[day]:.0f} cal {status}")
        print(f"           P:{daily_protein[day]:.0f}g({day_protein_ratio:.0f}%) | C:{daily_carbs[day]:.0f}g({day_carbs_ratio:.0f}%) | F:{daily_fat[day]:.0f}g({day_fat_ratio:.0f}%)")
    
    print(f"\n MEAL DIVERSITY:")
    print(f"  Unique meals: {len(unique_meals)} out of {len(solution) * 3}")
    
    print(f"\nMEAL PLAN:")
    for day in range(days):
        print(f"  Day {day+1}: {solution[day]}")
    print("="*60)



In [356]:
##### TESTING #####
test_search(2200, 7000, 7, strat="greedy")

A* MEAL PLAN RESULTS

 PARAMETERS:
  Daily TDEE target: 2200 cal
  Total budget: 7000 DZD
  Duration: 7 days

 COST ANALYSIS:
  Total cost: 5200.65 DZD
  Budget remaining: 1799.35 DZD (25.7%)
  Average daily cost: 742.95 DZD

 CALORIE ANALYSIS:
  Total calories: 13152 cal
  Expected calories: 15400 cal
  Deviation: -2248 cal (-14.6%)
  Average daily calories: 1879 cal

 MACRONUTRIENT DISTRIBUTION:
  Protein: 817 (24.9%) [Target: 30%] -5.1%
  Carbs:   1471 (44.7%) [Target: 45%] -0.3%
  Fat:     461 (31.5%) [Target: 25%] +6.5%

  Daily Avg Macros:
    Protein: 117g | Carbs: 210g | Fat: 66g

 DAILY BREAKDOWN:
  Day 1: 739.10 DZD, 1361 cal ⚠
           P:372g(27%) | C:325g(24%) | F:686g(50%)
  Day 2: 938.21 DZD, 1811 cal ⚠
           P:462g(26%) | C:741g(41%) | F:601g(33%)
  Day 3: 504.62 DZD, 1865 cal ⚠
           P:374g(20%) | C:899g(48%) | F:603g(32%)
  Day 4: 559.86 DZD, 2357 cal ✓
           P:547g(23%) | C:1224g(52%) | F:611g(26%)
  Day 5: 1006.97 DZD, 1716 cal ⚠
           P:534g(31